# Exercice 1 : CNN Variante VGG pour CIFAR-10
## TP2 — Reseaux de Neurones Convolutifs (PyTorch)

---

**Objectif :** Construire et entrainer un CNN inspire de VGG pour classifier des images CIFAR-10.

---

## Etape 1 — Activer le GPU

**Runtime > Change runtime type > GPU**

In [ ]:
import torch
print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")

## Etape 2 — Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

---
## Etape 3 — Charger CIFAR-10

**Dataset :**
- 60 000 images couleur 32x32
- 10 classes : avion, voiture, oiseau, chat, cerf, chien, grenouille, cheval, bateau, camion

In [ ]:
# Normalisation pour CIFAR-10
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

# Charger les donnees
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)

print(f"Train : {len(trainset)} images")
print(f"Test : {len(testset)} images")

In [ ]:
# Afficher quelques images
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# Denormaliser pour afficher
mean = torch.tensor([0.4914, 0.4822, 0.4465])
std = torch.tensor([0.2470, 0.2435, 0.2616])

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(10):
    ax = axes[i // 5, i % 5]
    img = trainset[i][0] * std[:, None, None] + mean[:, None, None]
    img = torch.clamp(img, 0, 1)
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(class_names[trainset[i][1]])
    ax.axis('off')
plt.tight_layout()
plt.show()

## Etape 4 — Pretraitement

In [ ]:
# Separer validation (20% du train)
train_size = int(0.8 * len(trainset))
val_size = len(trainset) - train_size
train_subset, val_subset = torch.utils.data.random_split(trainset, [train_size, val_size])

# Creer les DataLoaders
train_loader = torch.utils.data.DataLoader(train_subset, batch_size=64,
                                            shuffle=True, num_workers=2)
val_loader = torch.utils.data.DataLoader(val_subset, batch_size=64,
                                          shuffle=False, num_workers=2)
test_loader = torch.utils.data.DataLoader(testset, batch_size=64,
                                           shuffle=False, num_workers=2)

print(f"Train : {train_size} images")
print(f"Validation : {val_size} images")
print(f"Test : {len(testset)} images")

---
## Etape 5 — Construire le modele VGG-like

**Architecture :**
```
Bloc 1 : Conv(32) -> Conv(32) -> MaxPool -> Dropout
Bloc 2 : Conv(64) -> Conv(64) -> MaxPool -> Dropout
Bloc 3 : Conv(128) -> Conv(128) -> MaxPool -> Dropout
FC : Flatten -> Dense(512) -> Dropout -> Dense(10)
```

In [ ]:
class VGGNet(nn.Module):
    def __init__(self):
        super(VGGNet, self).__init__()

        # Bloc 1
        self.bloc1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )

        # Bloc 2
        self.bloc2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )

        # Bloc 3
        self.bloc3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )

        # Classification
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.bloc1(x)
        x = self.bloc2(x)
        x = self.bloc3(x)
        x = self.classifier(x)
        return x

model = VGGNet().to(device)
print(model)

## Etape 6 — Compiler et entrainer

In [ ]:
# Fonction de perte et optimiseur
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Historique
train_acc_history = []
val_acc_history = []
train_loss_history = []
val_loss_history = []

In [ ]:
# Entrainer
epochs = 20

for epoch in range(epochs):
    # --- Entrainement ---
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = correct / total
    train_loss = train_loss / len(train_loader)

    # --- Validation ---
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = correct / total
    val_loss = val_loss / len(val_loader)

    # Sauvegarder
    train_acc_history.append(train_acc)
    val_acc_history.append(val_acc)
    train_loss_history.append(train_loss)
    val_loss_history.append(val_loss)

    print(f"Epoch {epoch+1}/{epochs} — "
          f"Train Acc: {train_acc:.2%} — Val Acc: {val_acc:.2%}")

## Etape 7 — Evaluer

In [ ]:
# Evaluation sur le test set
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_acc = correct / total
print(f"\nTest accuracy : {test_acc:.2%}")

## Etape 8 — Courbes d'apprentissage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(train_acc_history, label='Train')
axes[0].plot(val_acc_history, label='Validation')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(train_loss_history, label='Train')
axes[1].plot(val_loss_history, label='Validation')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Resume

| Composant | Role |
|-----------|------|
| nn.Conv2d | Extraction de features |
| nn.MaxPool2d | Reduction de taille |
| nn.Dropout | Regularisation (evite le surapprentissage) |
| nn.Linear | Classification finale |
| nn.CrossEntropyLoss | Fonction de perte pour 10 classes |